# Train BERTimbau Large v7 — Cascata Hierárquica — SPR 2026

**Estratégia:** Classificação em cascata com 3 modelos separados.

## 🏗️ Arquitetura

```
Relatório
    │
    ▼
Modelo 1 (Binário)
    ├─ p_benigno ──► Modelo 2 → classe 1, 2 ou 3
    └─ p_alterado ─► Modelo 3 → classe 0, 4, 5 ou 6
```

| Modelo | Tarefa | Classes originais | Classes internas |
|--------|--------|-------------------|------------------|
| Modelo 1 | Binário | BENIGNO vs ALTERADO | 0, 1 |
| Modelo 2 | Benigno | 1, 2, 3 | 0, 1, 2 |
| Modelo 3 | Alterado | 0, 4, 5, 6 | 0, 1, 2, 3 |

**Inferência Soft (sem hard routing):**
```
final_prob[c] = p_benigno × prob_modelo2[c]   (c ∈ {1,2,3})
              + p_alterado × prob_modelo3[c]   (c ∈ {0,4,5,6})
```

## 🔑 Diferenças em relação ao v5/v6

| Parâmetro | v5 | v6 | v7 (este) |
|-----------|----|----|----------|
| Arquitetura | Flat 7-classes | Flat 7-classes | **Cascata 3 modelos** |
| SEED | 42 | 123 | **456** |
| Loss | Focal | Focal+Smooth | **Focal+Smooth** |
| Scheduler | Linear | Cosine | **Cosine** |
| Grad Accum | 1 | 2 | **2** |

## ⚙️ Configuração no Kaggle

| Setting | Valor |
|---------|-------|
| **Internet** | OFF |
| **Accelerator** | GPU T4 x2 (ou P100) |
| **Add Input** | Competition: `spr-2026-mammography-report-classification` |
| **Add Input** | Models → `bertimbau-ptbr-complete` (fabianofilho) |

## 📂 Output gerado

```
/kaggle/working/advanced_outputs_kaggle_7/
    ├── all_results.pkl
    ├── cascata_artifacts.pkl   # temperature, thresholds, mappings
    └── weights/
        ├── binary/             # fold_0.pt ... fold_4.pt
        │   ├── tokenizer/
        │   └── model_config/
        ├── benigno/            # fold_0.pt ... fold_4.pt
        │   └── model_config/
        └── alterado/           # fold_0.pt ... fold_4.pt
            └── model_config/
```

In [ ]:
import os
import re
import gc
import math
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    get_cosine_schedule_with_warmup,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from scipy.optimize import minimize_scalar
from tqdm.auto import tqdm

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════════
SEED             = 456       # diferente do v5=42 e v6=123
MAX_LEN          = 512
BATCH_SIZE       = 8
GRAD_ACCUM_STEPS = 2         # batch efetivo = 16
EPOCHS           = 5
LR               = 2e-5
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.1
LABEL_SMOOTHING  = 0.1
N_FOLDS          = 5
FOCAL_GAMMA      = 2.0
FOCAL_ALPHA      = 0.25
VERSION          = 7

# Mapeamento da cascata
# BENIGNO: classes BI-RADS 1, 2, 3 (negativo / benigno / prob. benigno)
# ALTERADO: classes BI-RADS 0, 4, 5, 6 (incompleto / suspeito / maligno)
CLASSES_BENIGNO = [1, 2, 3]
CLASSES_ALTERADO = [0, 4, 5, 6]

# Mapeamento: classe original → índice interno do modelo
BENIGNO_TO_IDX  = {c: i for i, c in enumerate(CLASSES_BENIGNO)}   # {1:0, 2:1, 3:2}
ALTERADO_TO_IDX = {c: i for i, c in enumerate(CLASSES_ALTERADO)}  # {0:0, 4:1, 5:2, 6:3}

# Paths
DATA_DIR    = Path('/kaggle/input/competitions/spr-2026-mammography-report-classification')
OUTPUT_BASE = Path(f'/kaggle/working/advanced_outputs_kaggle_{VERSION}')
W_BINARY    = OUTPUT_BASE / 'weights' / 'binary'
W_BENIGNO   = OUTPUT_BASE / 'weights' / 'benigno'
W_ALTERADO  = OUTPUT_BASE / 'weights' / 'alterado'
for d in [W_BINARY, W_BENIGNO, W_ALTERADO]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_CANDIDATES = [
    '/kaggle/input/models/fabianofilho/bertimbau-ptbr-complete/pytorch/default/1',
    '/kaggle/input/bertimbau-ptbr-complete/pytorch/bert-large-portuguese-cased/1',
    '/kaggle/input/bert-large-portuguese-cased',
]
MODEL_PATH = next((p for p in MODEL_CANDIDATES if Path(p).exists()), None)
assert MODEL_PATH, (
    'BERTimbau Large não encontrado!\n'
    'Adicione: Add Input → Models → bertimbau-ptbr-complete (fabianofilho)'
)

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_FP16 = torch.cuda.is_available()

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Device:     {DEVICE}')
print(f'FP16:       {USE_FP16}')
print(f'Model:      {MODEL_PATH}')
print(f'Output:     {OUTPUT_BASE}')
print(f'Seed:       {SEED}')
print(f'Benigno:    {CLASSES_BENIGNO} → índices internos 0..{len(CLASSES_BENIGNO)-1}')
print(f'Alterado:   {CLASSES_ALTERADO} → índices internos 0..{len(CLASSES_ALTERADO)-1}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CARREGAR DADOS
# ═══════════════════════════════════════════════════════════════════════════════
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train: {len(train_df)} amostras | Test: {len(test_df)} amostras')
print(f'\nDistribuição das classes:')
class_counts = train_df['target'].value_counts().sort_index()
for cls, cnt in class_counts.items():
    grupo = 'BENIGNO' if cls in CLASSES_BENIGNO else 'ALTERADO'
    rare  = ' ⚠️ RARA' if cls in [5, 6] else ''
    print(f'  Classe {cls} [{grupo}]: {cnt:5} ({cnt/len(train_df)*100:.1f}%){rare}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PRÉ-PROCESSAMENTO
# ═══════════════════════════════════════════════════════════════════════════════
INDICACAO_MARKERS   = ['indicação clínica:\n', 'indicação clínica:', 'indicação:', 'indicacao:']
ACHADOS_MARKERS     = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['análise comparativa:', 'analise comparativa:']

def extract_section(text, start_markers, end_markers=None):
    txt_lower = text.lower()
    start_idx = -1
    for m in start_markers:
        idx = txt_lower.find(m)
        if idx >= 0:
            start_idx = idx + len(m)
            break
    if start_idx < 0:
        return ''
    if end_markers is None:
        return text[start_idx:].strip()
    end_idx = len(text)
    for m in end_markers:
        idx = txt_lower.find(m, start_idx)
        if 0 < idx < end_idx:
            end_idx = idx
    return text[start_idx:end_idx].strip()

def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def build_input_text(report_text):
    report      = clean_text(report_text)
    indicacao   = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
    achados     = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS)
    if len(achados) < 10:
        return report
    parts = []
    if indicacao:   parts.append(f'Indicação: {indicacao}')
    if achados:     parts.append(f'Achados: {achados}')
    if comparativa: parts.append(f'Comparativa: {comparativa}')
    return ' '.join(parts)

all_texts    = [build_input_text(t) for t in train_df['report'].tolist()]
all_labels   = train_df['target'].tolist()
test_texts   = [build_input_text(t) for t in test_df['report'].tolist()]
print(f'Textos processados: {len(all_texts)} train, {len(test_texts)} test')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════════════════════════
class MammographyDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=512):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt',
        )
        item = {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in enc:
            item['token_type_ids'] = enc['token_type_ids'].squeeze(0)
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FOCAL LOSS + LABEL SMOOTHING
# ═══════════════════════════════════════════════════════════════════════════════
class FocalLossWithSmoothing(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, smoothing=0.1, num_classes=7):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
        self.smoothing, self.num_classes = smoothing, num_classes

    def forward(self, inputs, targets):
        with torch.no_grad():
            smooth_targets = torch.full_like(inputs, self.smoothing / (self.num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        log_probs = F.log_softmax(inputs, dim=-1)
        ce = -(smooth_targets * log_probs).sum(dim=-1)
        probs = F.softmax(inputs, dim=-1)
        pt    = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        return (self.alpha * (1 - pt) ** self.gamma * ce).mean()

print(f'FocalLossWithSmoothing: alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA}, smoothing={LABEL_SMOOTHING}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# HELPERS DE TREINO E AVALIAÇÃO
# ═══════════════════════════════════════════════════════════════════════════════
def make_criterion(num_classes):
    return FocalLossWithSmoothing(
        alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA,
        smoothing=LABEL_SMOOTHING, num_classes=num_classes
    )

def train_epoch(model, loader, optimizer, scheduler, scaler, criterion):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(loader, leave=False)):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)

        if USE_FP16:
            with torch.cuda.amp.autocast():
                loss = criterion(model(**kwargs).logits, labels) / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
        else:
            loss = criterion(model(**kwargs).logits, labels) / GRAD_ACCUM_STEPS
            loss.backward()

        total_loss += loss.item() * GRAD_ACCUM_STEPS
        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            if USE_FP16:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
    return total_loss / len(loader)


@torch.no_grad()
def get_probs(model, loader):
    model.eval()
    all_probs = []
    for batch in loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)
        all_probs.extend(F.softmax(model(**kwargs).logits, dim=-1).cpu().numpy())
    return np.array(all_probs)


def make_loader(texts, labels, tokenizer, shuffle=False):
    ds = MammographyDataset(texts, labels, tokenizer, MAX_LEN)
    return DataLoader(ds, batch_size=BATCH_SIZE if shuffle else BATCH_SIZE * 2,
                      shuffle=shuffle, num_workers=2, pin_memory=True)


def build_model(num_classes, weights_dir):
    config = AutoConfig.from_pretrained(MODEL_PATH, num_labels=num_classes)
    model  = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, config=config).to(DEVICE)
    config.save_pretrained(weights_dir / 'model_config')
    return model


def make_scheduler(optimizer, loader):
    steps_per_epoch = math.ceil(len(loader) / GRAD_ACCUM_STEPS)
    total_steps     = steps_per_epoch * EPOCHS
    warmup_steps    = int(WARMUP_RATIO * total_steps)
    return get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)


print('Helpers definidos.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CALIBRAÇÃO (no espaço 7 classes reconstruídas)
# ═══════════════════════════════════════════════════════════════════════════════
def temperature_scale(probs, temperature):
    logits     = np.log(probs + 1e-10)
    scaled     = logits / temperature
    exp_scaled = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return exp_scaled / exp_scaled.sum(axis=1, keepdims=True)

def apply_thresholds(probs, thresholds):
    return (probs * np.array(thresholds)).argmax(axis=1)

def optimize_temperature(oof_probs, oof_labels):
    def neg_f1(temp):
        return -f1_score(oof_labels, temperature_scale(oof_probs, temp).argmax(axis=1), average='macro')
    return minimize_scalar(neg_f1, bounds=(0.1, 5.0), method='bounded').x

def optimize_thresholds(cal_probs, oof_labels, n_steps=50):
    best_thresholds = [1.0] * 7
    best_f1 = f1_score(oof_labels, cal_probs.argmax(axis=1), average='macro')
    grid = np.linspace(0.05, 3.0, n_steps)
    for c in range(7):
        best_t_c = best_thresholds[c]
        for t in grid:
            thresh_try    = best_thresholds.copy()
            thresh_try[c] = t
            f1 = f1_score(oof_labels, apply_thresholds(cal_probs, thresh_try), average='macro')
            if f1 > best_f1:
                best_f1, best_t_c = f1, t
        best_thresholds[c] = best_t_c
    return best_thresholds, best_f1


def soft_cascade(binary_probs, benigno_probs, alterado_probs):
    """
    Reconstrói 7 probabilidades finais via soft cascade:
        final[c] = p_benigno * prob_benigno[c]   para c in {1,2,3}
                 + p_alterado * prob_alterado[c]  para c in {0,4,5,6}

    Args:
        binary_probs:   (N, 2)  — [p_benigno, p_alterado]
        benigno_probs:  (N, 3)  — probs para classes internas 0,1,2 → originais 1,2,3
        alterado_probs: (N, 4)  — probs para classes internas 0,1,2,3 → originais 0,4,5,6
    Returns:
        final_probs:    (N, 7)  — probs para classes 0-6
    """
    N = len(binary_probs)
    p_ben = binary_probs[:, 0:1]   # (N, 1)
    p_alt = binary_probs[:, 1:2]   # (N, 1)

    final = np.zeros((N, 7))
    # Benigno: classes 1,2,3
    for i, c in enumerate(CLASSES_BENIGNO):
        final[:, c] = (p_ben * benigno_probs[:, i:i+1]).squeeze(1)
    # Alterado: classes 0,4,5,6
    for i, c in enumerate(CLASSES_ALTERADO):
        final[:, c] = (p_alt * alterado_probs[:, i:i+1]).squeeze(1)

    # Normalizar (soma = 1 por construção, mas evita acúmulo de float)
    final = final / final.sum(axis=1, keepdims=True)
    return final

print('Calibração e soft cascade definidos.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TOKENIZER (compartilhado pelos 3 modelos)
# ═══════════════════════════════════════════════════════════════════════════════
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.save_pretrained(W_BINARY / 'tokenizer')
print(f'Tokenizer salvo em {W_BINARY / "tokenizer"}')

# Arrays para o loop de k-fold
train_arr  = np.array(all_texts)
label_arr  = np.array(all_labels)
label_binary = np.array([0 if c in CLASSES_BENIGNO else 1 for c in all_labels])  # 0=ben,1=alt

# Test loaders (estáticos, usados para inferência em todos os folds)
test_loader_full = make_loader(test_texts, None, tokenizer)

# Acumuladores OOF e test_probs para cada modelo
oof_binary  = np.zeros((len(train_df), 2))
oof_benigno = np.zeros((len(train_df), len(CLASSES_BENIGNO)))
oof_alterado= np.zeros((len(train_df), len(CLASSES_ALTERADO)))

test_binary  = np.zeros((len(test_df), 2))
test_benigno = np.zeros((len(test_df), len(CLASSES_BENIGNO)))
test_alterado= np.zeros((len(test_df), len(CLASSES_ALTERADO)))

all_results = {}
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
print(f'K-Fold: {N_FOLDS} splits')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LOOP 5-FOLD — TREINA OS 3 MODELOS POR FOLD
# ═══════════════════════════════════════════════════════════════════════════════
for fold, (tr_idx, vl_idx) in enumerate(skf.split(train_arr, label_arr)):
    print(f'\n{"="*65}')
    print(f'FOLD {fold+1}/{N_FOLDS}')
    print(f'{"="*65}')

    X_tr, y_tr = train_arr[tr_idx].tolist(), label_arr[tr_idx].tolist()
    X_vl, y_vl = train_arr[vl_idx].tolist(), label_arr[vl_idx].tolist()
    y_bin_tr   = label_binary[tr_idx].tolist()
    y_bin_vl   = label_binary[vl_idx].tolist()

    # ────────────────────────────────────────────────────────────────
    # MODELO 1: Binário (BENIGNO vs ALTERADO) — treina em todos os dados
    # ────────────────────────────────────────────────────────────────
    print(f'\n--- Modelo 1: Binário (2 classes) ---')
    m1 = build_model(2, W_BINARY)
    opt1 = torch.optim.AdamW(m1.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    tr_ldr1 = make_loader(X_tr, y_bin_tr, tokenizer, shuffle=True)
    vl_ldr1 = make_loader(X_vl, y_bin_vl, tokenizer)
    sch1    = make_scheduler(opt1, tr_ldr1)
    scaler1 = torch.cuda.amp.GradScaler() if USE_FP16 else None
    crit1   = make_criterion(2)
    best_f1_1, best_state_1 = 0.0, None

    for ep in range(EPOCHS):
        loss = train_epoch(m1, tr_ldr1, opt1, sch1, scaler1, crit1)
        vp = get_probs(m1, vl_ldr1)
        f1 = f1_score(y_bin_vl, vp.argmax(axis=1), average='macro')
        print(f'  M1 ep{ep+1} loss={loss:.4f} f1={f1:.5f}')
        if f1 > best_f1_1:
            best_f1_1   = f1
            best_state_1 = {k: v.cpu().clone() for k, v in m1.state_dict().items()}

    torch.save(best_state_1, W_BINARY / f'fold_{fold}.pt')
    m1.load_state_dict({k: v.to(DEVICE) for k, v in best_state_1.items()})
    oof_binary[vl_idx]  = get_probs(m1, vl_ldr1)
    test_binary         += get_probs(m1, test_loader_full)
    del m1, best_state_1, tr_ldr1, vl_ldr1
    gc.collect(); torch.cuda.empty_cache()

    # ────────────────────────────────────────────────────────────────
    # MODELO 2: Benigno (classes 1, 2, 3) — treina só no subset benigno
    # ────────────────────────────────────────────────────────────────
    print(f'\n--- Modelo 2: Benigno (3 classes: {CLASSES_BENIGNO}) ---')
    mask_tr_ben = [i for i, c in enumerate(y_tr) if c in CLASSES_BENIGNO]
    mask_vl_ben = [i for i, c in enumerate(y_vl) if c in CLASSES_BENIGNO]
    X_tr2 = [X_tr[i] for i in mask_tr_ben]
    y_tr2 = [BENIGNO_TO_IDX[y_tr[i]] for i in mask_tr_ben]
    X_vl2 = [X_vl[i] for i in mask_vl_ben]
    y_vl2 = [BENIGNO_TO_IDX[y_vl[i]] for i in mask_vl_ben]
    vl_ben_orig = [y_vl[i] for i in mask_vl_ben]  # para OOF
    vl_ben_global_idx = vl_idx[mask_vl_ben]        # índice global no train_df
    print(f'  Train benigno: {len(X_tr2)} | Val benigno: {len(X_vl2)}')

    m2 = build_model(len(CLASSES_BENIGNO), W_BENIGNO)
    opt2 = torch.optim.AdamW(m2.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    tr_ldr2 = make_loader(X_tr2, y_tr2, tokenizer, shuffle=True)
    vl_ldr2 = make_loader(X_vl2, y_vl2, tokenizer)
    sch2    = make_scheduler(opt2, tr_ldr2)
    scaler2 = torch.cuda.amp.GradScaler() if USE_FP16 else None
    crit2   = make_criterion(len(CLASSES_BENIGNO))
    best_f1_2, best_state_2 = 0.0, None

    for ep in range(EPOCHS):
        loss = train_epoch(m2, tr_ldr2, opt2, sch2, scaler2, crit2)
        vp = get_probs(m2, vl_ldr2)
        f1 = f1_score(y_vl2, vp.argmax(axis=1), average='macro')
        print(f'  M2 ep{ep+1} loss={loss:.4f} f1={f1:.5f}')
        if f1 > best_f1_2:
            best_f1_2    = f1
            best_state_2 = {k: v.cpu().clone() for k, v in m2.state_dict().items()}

    torch.save(best_state_2, W_BENIGNO / f'fold_{fold}.pt')
    m2.load_state_dict({k: v.to(DEVICE) for k, v in best_state_2.items()})
    oof_benigno[vl_ben_global_idx] = get_probs(m2, vl_ldr2)
    test_benigno += get_probs(m2, test_loader_full)
    del m2, best_state_2, tr_ldr2, vl_ldr2
    gc.collect(); torch.cuda.empty_cache()

    # ────────────────────────────────────────────────────────────────
    # MODELO 3: Alterado (classes 0, 4, 5, 6) — treina só no subset alterado
    # ────────────────────────────────────────────────────────────────
    print(f'\n--- Modelo 3: Alterado (4 classes: {CLASSES_ALTERADO}) ---')
    mask_tr_alt = [i for i, c in enumerate(y_tr) if c in CLASSES_ALTERADO]
    mask_vl_alt = [i for i, c in enumerate(y_vl) if c in CLASSES_ALTERADO]
    X_tr3 = [X_tr[i] for i in mask_tr_alt]
    y_tr3 = [ALTERADO_TO_IDX[y_tr[i]] for i in mask_tr_alt]
    X_vl3 = [X_vl[i] for i in mask_vl_alt]
    y_vl3 = [ALTERADO_TO_IDX[y_vl[i]] for i in mask_vl_alt]
    vl_alt_global_idx = vl_idx[mask_vl_alt]
    print(f'  Train alterado: {len(X_tr3)} | Val alterado: {len(X_vl3)}')

    m3 = build_model(len(CLASSES_ALTERADO), W_ALTERADO)
    opt3 = torch.optim.AdamW(m3.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    tr_ldr3 = make_loader(X_tr3, y_tr3, tokenizer, shuffle=True)
    vl_ldr3 = make_loader(X_vl3, y_vl3, tokenizer)
    sch3    = make_scheduler(opt3, tr_ldr3)
    scaler3 = torch.cuda.amp.GradScaler() if USE_FP16 else None
    crit3   = make_criterion(len(CLASSES_ALTERADO))
    best_f1_3, best_state_3 = 0.0, None

    for ep in range(EPOCHS):
        loss = train_epoch(m3, tr_ldr3, opt3, sch3, scaler3, crit3)
        vp = get_probs(m3, vl_ldr3)
        f1 = f1_score(y_vl3, vp.argmax(axis=1), average='macro')
        print(f'  M3 ep{ep+1} loss={loss:.4f} f1={f1:.5f}')
        if f1 > best_f1_3:
            best_f1_3    = f1
            best_state_3 = {k: v.cpu().clone() for k, v in m3.state_dict().items()}

    torch.save(best_state_3, W_ALTERADO / f'fold_{fold}.pt')
    m3.load_state_dict({k: v.to(DEVICE) for k, v in best_state_3.items()})
    oof_alterado[vl_alt_global_idx] = get_probs(m3, vl_ldr3)
    test_alterado += get_probs(m3, test_loader_full)
    del m3, best_state_3, tr_ldr3, vl_ldr3
    gc.collect(); torch.cuda.empty_cache()

    # ── Salva por fold para inspeção
    all_results[f'fold_{fold}'] = {
        'val_idx':    vl_idx.tolist(),
        'val_labels': y_vl,
        'f1_binary':  float(best_f1_1),
        'f1_benigno': float(best_f1_2),
        'f1_alterado':float(best_f1_3),
    }
    print(f'\n✓ Fold {fold} | M1_f1={best_f1_1:.4f} M2_f1={best_f1_2:.4f} M3_f1={best_f1_3:.4f}')


# Média dos folds nos test_probs
test_binary   /= N_FOLDS
test_benigno  /= N_FOLDS
test_alterado /= N_FOLDS
print('\n✅ Treino concluído.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# RECONSTRUÇÃO OOF — soft cascade
# ═══════════════════════════════════════════════════════════════════════════════
# Para amostras BENIGNAS no OOF: oof_benigno foi preenchido
# Para amostras ALTERADAS no OOF: oof_alterado foi preenchido
# Amostras benignas têm oof_alterado=0 e vice-versa → ok para soft_cascade

# Preencher zeros com distribuição uniforme para evitar divisão por zero
# (cada amostra foi classificada por apenas um dos dois sub-modelos)
oof_benigno_filled  = oof_benigno.copy()
oof_alterado_filled = oof_alterado.copy()

for i, c in enumerate(all_labels):
    if c in CLASSES_BENIGNO:
        # oof_benigno está preenchido; oof_alterado preenche uniforme
        oof_alterado_filled[i] = 1.0 / len(CLASSES_ALTERADO)
    else:
        # oof_alterado está preenchido; oof_benigno preenche uniforme
        oof_benigno_filled[i]  = 1.0 / len(CLASSES_BENIGNO)

oof_final = soft_cascade(oof_binary, oof_benigno_filled, oof_alterado_filled)
oof_f1    = f1_score(all_labels, oof_final.argmax(axis=1), average='macro')

print(f'OOF F1-macro (cascata):    {oof_f1:.5f}')
print(f'OOF F1-macro (só binário): {f1_score(label_binary, oof_binary.argmax(axis=1), average="macro"):.5f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CALIBRAÇÃO — temperatura + thresholds no espaço de 7 classes
# ═══════════════════════════════════════════════════════════════════════════════
print('Otimizando temperatura...')
best_temp      = optimize_temperature(oof_final, label_arr)
oof_calibrated = temperature_scale(oof_final, best_temp)
f1_after_temp  = f1_score(all_labels, oof_calibrated.argmax(axis=1), average='macro')
print(f'  Temperatura: {best_temp:.4f} | F1: {f1_after_temp:.5f}')

print('Otimizando thresholds...')
best_thresholds, f1_final = optimize_thresholds(oof_calibrated, label_arr)
print(f'  Thresholds: {[round(t, 4) for t in best_thresholds]}')
print(f'  F1 final:   {f1_final:.5f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SALVAR ARTIFACTS
# ═══════════════════════════════════════════════════════════════════════════════
cascata_artifacts = {
    'temperature':       best_temp,
    'thresholds':        best_thresholds,
    'oof_f1_macro':      oof_f1,
    'oof_f1_calibrated': f1_final,
    'classes_benigno':   CLASSES_BENIGNO,
    'classes_alterado':  CLASSES_ALTERADO,
    'benigno_to_idx':    BENIGNO_TO_IDX,
    'alterado_to_idx':   ALTERADO_TO_IDX,
}
with open(OUTPUT_BASE / 'cascata_artifacts.pkl', 'wb') as f:
    pickle.dump(cascata_artifacts, f)

with open(OUTPUT_BASE / 'all_results.pkl', 'wb') as f:
    pickle.dump(all_results, f)

print('Artifacts salvos.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# INFERÊNCIA FINAL + SUBMISSION
# ═══════════════════════════════════════════════════════════════════════════════
test_final   = soft_cascade(test_binary, test_benigno, test_alterado)
test_cal     = temperature_scale(test_final, best_temp)
predictions  = apply_thresholds(test_cal, best_thresholds)

submission = pd.DataFrame({'ID': test_df['ID'], 'target': predictions})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print('=== SUBMISSION ===')
print(submission['target'].value_counts().sort_index().to_string())
print(f'\n{len(submission)} linhas salvas.')

# Verificar estrutura do output
print(f'\nEstrutura do output:')
total = 0
for root, dirs, files in os.walk(OUTPUT_BASE):
    dirs.sort()
    indent = '  ' * len(Path(root).relative_to(OUTPUT_BASE).parts)
    print(f'{indent}{Path(root).name}/')
    for fname in sorted(files):
        fpath = Path(root) / fname
        size  = fpath.stat().st_size
        total += size
        print(f'{indent}  {fname}  ({size/1024/1024:.1f} MB)')
print(f'\nTotal: {total/1024/1024:.0f} MB')

print(f'\n✅ Treino v7 (cascata) concluído!')
print(f'   OOF F1 raw:         {oof_f1:.5f}')
print(f'   OOF F1 calibrado:   {f1_final:.5f}')
print(f'   Temperatura:        {best_temp:.4f}')
print(f'   Thresholds:         {[round(t,4) for t in best_thresholds]}')
print(f'\n➡️  Próximos passos:')
print(f'   1. Save Version → Save & Run All (Commit)')
print(f'   2. Output → New Dataset → nome: model-v{VERSION}')
print(f'   3. Criar notebook resubmit_cascata_v7 para submissão')